In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-addon-obp qiskit-addon-utils rustworkx
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

*ประมาณเวลาการใช้งาน: 16 นาทีบนโปรเซสเซอร์ Eagle r3 (หมายเหตุ: นี่เป็นเพียงการประมาณการ เวลาจริงอาจแตกต่างกันได้)*

In [ ]:
# This cell is hidden from users;
# it disables linting rules.
# ruff: noqa

## พื้นหลัง

การย้อนกลับตัวดำเนินการ (Operator backpropagation) คือเทคนิคที่ดูดซับการดำเนินการจากส่วนท้ายของ Circuit ควอนตัมเข้าสู่ observable ที่วัดได้ ซึ่งโดยทั่วไปจะลดความลึกของ Circuit แลกกับจำนวน term ที่เพิ่มขึ้นใน observable เป้าหมายคือย้อนกลับ Circuit ให้ได้มากที่สุดเท่าที่จะทำได้โดยไม่ให้ observable ขยายใหญ่เกินไป การนำไปใช้งานบน Qiskit มีอยู่ใน OBP Qiskit addon โดยสามารถดูรายละเอียดเพิ่มเติมได้ที่ [เอกสาร](/guides/qiskit-addons-obp) พร้อม [ตัวอย่างง่าย ๆ](/guides/qiskit-addons-obp-get-started) สำหรับการเริ่มต้น

พิจารณา Circuit ตัวอย่างที่ต้องวัด observable $O = \sum_P c_P P$ โดยที่ $P$ คือ Pauli และ $c_P$ คือสัมประสิทธิ์ กำหนดให้ Circuit แทนด้วย unitary เดี่ยว $U$ ซึ่งสามารถแบ่งแบบ logical ได้เป็น $U = U_C U_Q$ ดังแสดงในรูปด้านล่าง

![Circuit diagram showing Uq followed by Uc](../docs/images/tutorials/improving-estimation-of-expectation-values-with-operator-backpropagation/logical-partitioning.avif)

การย้อนกลับตัวดำเนินการดูดซับ unitary $U_C$ เข้าสู่ observable โดยวิวัฒนาการมันเป็น $O' = U_C^{\dagger}OU_C = \sum_P c_P U_C^{\dagger}PU_C$ พูดอีกนัยหนึ่งคือ ส่วนหนึ่งของการคำนวณดำเนินการแบบ classical ผ่านการวิวัฒนาการของ observable จาก $O$ ไปเป็น $O'$ ปัญหาเดิมสามารถกำหนดใหม่ได้เป็นการวัด observable $O'$ สำหรับ Circuit ใหม่ที่มีความลึกน้อยลง ซึ่งมี unitary เป็น $U_Q$

Unitary $U_C$ แทนด้วยจำนวน slice $U_C = U_S U_{S-1}...U_2U_1$ มีหลายวิธีในการกำหนด slice ตัวอย่างเช่น ใน Circuit ตัวอย่างข้างต้น แต่ละเลเยอร์ของ $R_{zz}$ และแต่ละเลเยอร์ของ Gate $R_x$ สามารถถือเป็น slice ได้ การย้อนกลับเกี่ยวข้องกับการคำนวณ $O' = \Pi_{s=1}^S \sum_P c_P U_s^{\dagger} P U_s$ แบบ classical โดย slice แต่ละอัน $U_s$ สามารถแทนได้เป็น $U_s = exp(\frac{-i\theta_s P_s}{2})$ โดยที่ $P_s$ คือ Pauli ขนาด $n$ Qubit และ $\theta_s$ คือสเกลาร์ สามารถตรวจสอบได้ง่ายว่า

$$
U_s^{\dagger} P U_s = P \qquad \text{if} ~[P,P_s] = 0,
$$
$$
U_s^{\dagger} P U_s = \qquad cos(\theta_s)P + i sin(\theta_s)P_sP \qquad \text{if} ~{P,P_s} = 0
$$

ในตัวอย่างข้างต้น ถ้า ${P,P_s} = 0$ เราต้องรัน Circuit ควอนตัมสอง Circuit แทนที่จะเป็นหนึ่ง Circuit เพื่อคำนวณค่าความคาดหวัง ดังนั้น การย้อนกลับอาจเพิ่มจำนวน term ใน observable ทำให้ต้องรัน Circuit มากขึ้น วิธีหนึ่งที่จะอนุญาตให้ย้อนกลับลึกกว่าเข้าไปใน Circuit โดยไม่ให้ตัวดำเนินการขยายใหญ่เกินไปคือการตัดทอน term ที่มีสัมประสิทธิ์เล็ก แทนที่จะเพิ่มลงใน operator ตัวอย่างเช่น ในตัวอย่างข้างต้น อาจเลือกตัด term ที่เกี่ยวกับ $P_sP$ ออก หากว่า $\theta_s$ เล็กพอ การตัดทอน term อาจส่งผลให้ต้องรัน Circuit ควอนตัมน้อยลง แต่จะเกิดข้อผิดพลาดบางส่วนในการคำนวณค่าความคาดหวังสุดท้ายซึ่งสัดส่วนกับขนาดของสัมประสิทธิ์ของ term ที่ถูกตัดออก

บทเรียนนี้นำ [รูปแบบ Qiskit](/guides/intro-to-patterns) มาใช้จำลองพลศาสตร์ควอนตัมของโซ่สปิน Heisenberg โดยใช้ <a href="https://github.com/Qiskit/qiskit-addon-obp">qiskit-addon-obp</a>

## ข้อกำหนดเบื้องต้น

ก่อนเริ่มบทเรียนนี้ ให้แน่ใจว่าได้ติดตั้งสิ่งต่อไปนี้แล้ว:

- Qiskit SDK v1.2 หรือใหม่กว่า (`pip install qiskit`)
- Qiskit Runtime v0.28 หรือใหม่กว่า (`pip install qiskit-ibm-runtime`)
- OBP Qiskit addon (`pip install qiskit-addon-obp`)
- Qiskit addon utils (`pip install qiskit-addon-utils`)

## การตั้งค่า

In [2]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit.primitives import StatevectorEstimator as Estimator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import CouplingMap
from qiskit.synthesis import LieTrotter

from qiskit_addon_utils.problem_generators import generate_xyz_hamiltonian
from qiskit_addon_utils.problem_generators import (
    generate_time_evolution_circuit,
)
from qiskit_addon_utils.slicing import slice_by_gate_types, combine_slices
from qiskit_addon_obp.utils.simplify import OperatorBudget
from qiskit_addon_obp import backpropagate
from qiskit_addon_obp.utils.truncating import setup_budget

from rustworkx.visualization import graphviz_draw

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2, EstimatorOptions

## ส่วนที่ I: โซ่สปิน Heisenberg ขนาดเล็ก
### ขั้นตอนที่ 1: แมป input แบบ classical ไปสู่ปัญหาควอนตัม
#### แมปการวิวัฒนาการตามเวลาของโมเดลควอนตัม Heisenberg ไปสู่การทดลองควอนตัม
แพ็กเกจ [qiskit_addon_utils](https://github.com/Qiskit/qiskit-addon-utils) มีฟังก์ชันที่สามารถนำกลับมาใช้ใหม่ได้สำหรับวัตถุประสงค์ต่าง ๆ

โมดูล [qiskit_addon_utils.problem_generators](https://docs.quantum.ibm.com/api/qiskit-addon-utils/problem-generators) มีฟังก์ชันสำหรับสร้าง Hamiltonian แบบ Heisenberg บนกราฟการเชื่อมต่อที่กำหนด
กราฟนี้สามารถเป็น [rustworkx.PyGraph](https://www.rustworkx.org/apiref/rustworkx.PyGraph.html) หรือ [CouplingMap](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.CouplingMap) ซึ่งใช้งานได้ง่ายในขั้นตอนงานที่เน้น Qiskit

ในส่วนต่อไปนี้ เราจะสร้าง `CouplingMap` แบบโซ่เชิงเส้นที่มี 10 Qubit

In [3]:
num_qubits = 10
layout = [(i - 1, i) for i in range(1, num_qubits)]

# Instantiate a CouplingMap object
coupling_map = CouplingMap(layout)
graphviz_draw(coupling_map.graph, method="circo")

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/de8ce35e-a2c5-474f-adb9-88c9afb2bd76-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/operator-back-propagation/extracted-outputs/de8ce35e-a2c5-474f-adb9-88c9afb2bd76-0.avif)

ต่อไป เราจะสร้าง Pauli operator ที่จำลอง Hamiltonian XYZ แบบ Heisenberg

$$
{\hat{\mathcal{H}}_{XYZ} = \sum_{(j,k)\in E} (J_{x} \sigma_j^{x} \sigma_{k}^{x} + J_{y} \sigma_j^{y} \sigma_{k}^{y} + J_{z} \sigma_j^{z} \sigma_{k}^{z}) + \sum_{j\in V} (h_{x} \sigma_j^{x} + h_{y} \sigma_j^{y} + h_{z} \sigma_j^{z})}
$$

โดยที่ $G(V,E)$ คือกราฟของ coupling map ที่ให้มา

In [4]:
# Get a qubit operator describing the Heisenberg XYZ model
hamiltonian = generate_xyz_hamiltonian(
    coupling_map,
    coupling_constants=(np.pi / 8, np.pi / 4, np.pi / 2),
    ext_magnetic_field=(np.pi / 3, np.pi / 6, np.pi / 9),
)
print(hamiltonian)

SparsePauliOp(['IIIIIIIXXI', 'IIIIIIIYYI', 'IIIIIIIZZI', 'IIIIIXXIII', 'IIIIIYYIII', 'IIIIIZZIII', 'IIIXXIIIII', 'IIIYYIIIII', 'IIIZZIIIII', 'IXXIIIIIII', 'IYYIIIIIII', 'IZZIIIIIII', 'IIIIIIIIXX', 'IIIIIIIIYY', 'IIIIIIIIZZ', 'IIIIIIXXII', 'IIIIIIYYII', 'IIIIIIZZII', 'IIIIXXIIII', 'IIIIYYIIII', 'IIIIZZIIII', 'IIXXIIIIII', 'IIYYIIIIII', 'IIZZIIIIII', 'XXIIIIIIII', 'YYIIIIIIII', 'ZZIIIIIIII', 'IIIIIIIIIX', 'IIIIIIIIIY', 'IIIIIIIIIZ', 'IIIIIIIIXI', 'IIIIIIIIYI', 'IIIIIIIIZI', 'IIIIIIIXII', 'IIIIIIIYII', 'IIIIIIIZII', 'IIIIIIXIII', 'IIIIIIYIII', 'IIIIIIZIII', 'IIIIIXIIII', 'IIIIIYIIII', 'IIIIIZIIII', 'IIIIXIIIII', 'IIIIYIIIII', 'IIIIZIIIII', 'IIIXIIIIII', 'IIIYIIIIII', 'IIIZIIIIII', 'IIXIIIIIII', 'IIYIIIIIII', 'IIZIIIIIII', 'IXIIIIIIII', 'IYIIIIIIII', 'IZIIIIIIII', 'XIIIIIIIII', 'YIIIIIIIII', 'ZIIIIIIIII'],
              coeffs=[0.39269908+0.j, 0.78539816+0.j, 1.57079633+0.j, 0.39269908+0.j,
 0.78539816+0.j, 1.57079633+0.j, 0.39269908+0.j, 0.78539816+0.j,
 1.57079633+0.j, 0.39269908+0.j, 0.

From the qubit operator, we can generate a quantum circuit which models its time evolution.
Once again, the [qiskit_addon_utils.problem_generators](/docs/api/qiskit-addon-utils/problem-generators) module comes to the rescue with a handy function do just that:

In [5]:
circuit = generate_time_evolution_circuit(
    hamiltonian,
    time=0.2,
    synthesis=LieTrotter(reps=2),
)
circuit.draw("mpl", style="iqp", scale=0.6)

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/1d68f197-ffa4-49de-9fe8-243b1facbd00-0.avif" alt="Output of the previous code cell" />

จาก qubit operator เราสามารถสร้าง Circuit ควอนตัมที่จำลองการวิวัฒนาการตามเวลาของมันได้
อีกครั้ง โมดูล [qiskit_addon_utils.problem_generators](https://docs.quantum.ibm.com/api/qiskit-addon-utils/problem-generators) มีฟังก์ชันที่สะดวกสำหรับทำสิ่งนี้:

In [6]:
slices = slice_by_gate_types(circuit)
print(f"Separated the circuit into {len(slices)} slices.")

Separated the circuit into 18 slices.


![Output of the previous code cell](../docs/images/tutorials/operator-back-propagation/extracted-outputs/1d68f197-ffa4-49de-9fe8-243b1facbd00-0.avif)

### ขั้นตอนที่ 2: ปรับปัญหาให้เหมาะสมสำหรับการรันบน hardware ควอนตัม
#### สร้าง slice ของ Circuit สำหรับการย้อนกลับ
จำไว้ว่าฟังก์ชัน ``backpropagate`` จะย้อนกลับ Circuit ทีละ slice ดังนั้นการเลือกวิธีตัด slice อาจมีผลต่อประสิทธิภาพของการย้อนกลับสำหรับปัญหาที่กำหนด ที่นี่เราจะจัดกลุ่ม Gate ประเภทเดียวกันเป็น slice โดยใช้ฟังก์ชัน [slice_by_gate_types](https://docs.quantum.ibm.com/api/qiskit-addon-utils/slicing#slice_by_gate_types)

สำหรับการอภิปรายเพิ่มเติมเกี่ยวกับการตัด Circuit เป็น slice ดู [how-to guide](https://qiskit.github.io/qiskit-addon-utils/how_tos/create_circuit_slices.html) ของแพ็กเกจ [qiskit-addon-utils](https://github.com/Qiskit/qiskit-addon-utils)

In [7]:
op_budget = OperatorBudget(max_qwc_groups=8)

#### Backpropagate slices from the circuit

First we specify the observable to be $M_Z = \frac{1}{N} \sum_{i=1}^N \langle Z_i \rangle$, $N$ being the number of qubits. We will backpropagate slices from the time-evolution circuit until the terms in the observable can no longer be combined into eight or fewer qubit-wise commuting Pauli groups.

In [8]:
observable = SparsePauliOp.from_sparse_list(
    [("Z", [i], 1 / num_qubits) for i in range(num_qubits)],
    num_qubits=num_qubits,
)
observable

SparsePauliOp(['IIIIIIIIIZ', 'IIIIIIIIZI', 'IIIIIIIZII', 'IIIIIIZIII', 'IIIIIZIIII', 'IIIIZIIIII', 'IIIZIIIIII', 'IIZIIIIIII', 'IZIIIIIIII', 'ZIIIIIIIII'],
              coeffs=[0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j,
 0.1+0.j, 0.1+0.j])

#### จำกัดขนาดที่ operator อาจเติบโตในระหว่างการย้อนกลับ
ในระหว่างการย้อนกลับ จำนวน term ใน operator โดยทั่วไปจะเข้าใกล้ $4^N$ อย่างรวดเร็ว โดยที่ $N$ คือจำนวน Qubit เมื่อ term สอง term ใน operator ไม่ commute กันแบบ qubit-wise เราต้องการ Circuit แยกกันเพื่อหาค่าความคาดหวังที่สอดคล้องกัน ตัวอย่างเช่น ถ้ามี observable 2 Qubit $O = 0.1 XX + 0.3 IZ - 0.5 IX$ เนื่องจาก $[XX,IX] = 0$ การวัดในฐานเดียวก็เพียงพอเพื่อคำนวณค่าความคาดหวังสำหรับ term สองนั้น อย่างไรก็ตาม $IZ$ anti-commute กับอีกสอง term ดังนั้นเราต้องการการวัดฐานแยกต่างหากเพื่อคำนวณค่าความคาดหวังของ $IZ$ พูดอีกนัยหนึ่งคือ เราต้องการสอง Circuit แทนที่จะเป็นหนึ่ง Circuit เพื่อคำนวณ $\langle O \rangle$ เมื่อจำนวน term ใน operator เพิ่มขึ้น จำนวนการรัน Circuit ที่ต้องการก็อาจเพิ่มขึ้นด้วย

ขนาดของ operator สามารถจำกัดได้โดยระบุ kwarg ``operator_budget`` ของฟังก์ชัน ``backpropagate`` ซึ่งรับ instance ของ [OperatorBudget](https://docs.quantum.ibm.com/api/qiskit-addon-obp/utils-simplify#operatorbudget)

เพื่อควบคุมปริมาณทรัพยากรเพิ่มเติม (เวลา) ที่จัดสรร เราจำกัดจำนวน qubit-wise commuting Pauli group สูงสุดที่ observable ที่ย้อนกลับแล้วจะมีได้ ที่นี่เราระบุว่าการย้อนกลับควรหยุดเมื่อจำนวน qubit-wise commuting Pauli group ใน operator เกิน 8

In [9]:
# Backpropagate slices onto the observable
bp_obs, remaining_slices, metadata = backpropagate(
    observable, slices, operator_budget=op_budget
)
# Recombine the slices remaining after backpropagation
bp_circuit = combine_slices(remaining_slices)

print(f"Backpropagated {metadata.num_backpropagated_slices} slices.")
print(
    f"New observable has {len(bp_obs.paulis)} terms, which can be combined into {len(bp_obs.group_commuting(qubit_wise=True))} groups."
)
print(
    f"Note that backpropagating one more slice would result in {metadata.backpropagation_history[-1].num_paulis[0]} terms "
    f"across {metadata.backpropagation_history[-1].num_qwc_groups} groups."
)
print("The remaining circuit after backpropagation looks as follows:")
bp_circuit.draw("mpl", fold=-1, scale=0.6)

Backpropagated 6 slices.
New observable has 60 terms, which can be combined into 6 groups.
Note that backpropagating one more slice would result in 114 terms across 12 groups.
The remaining circuit after backpropagation looks as follows:


<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/65ec9cb1-a4ed-497b-a616-180e9659956f-1.avif" alt="Output of the previous code cell" />

#### ย้อนกลับ slice จาก Circuit
ก่อนอื่นเราระบุให้ observable เป็น $M_Z = \frac{1}{N} \sum_{i=1}^N \langle Z_i \rangle$ โดยที่ $N$ คือจำนวน Qubit เราจะย้อนกลับ slice จาก Circuit การวิวัฒนาการตามเวลาจนกว่า term ใน observable จะไม่สามารถรวมเป็นแปดกลุ่มหรือน้อยกว่าของ qubit-wise commuting Pauli group ได้อีกต่อไป

In [10]:
truncation_error_budget = setup_budget(max_error_per_slice=0.005)

Note that by allocating `5e-3` error per slice for truncation, we are able to remove 1 more slice from the circuit, while remaining within the original budget of eight commuting Pauli groups in the observable. By default, `backpropagate` uses the L1 norm of the truncated coefficients to bound the total error incurred from truncation. For other options refer to the [how-to guide on specifying the p_norm](https://qiskit.github.io/qiskit-addon-obp/how_tos/bound_error_using_p_norm.html).

In this particular example where we have backpropagated seven slices, the total truncation error should not exceed ``(5e-3 error/slice) * (7 slices) = 3.5e-2``.
For further discussion on distributing an error budget across your slices, check out [this how-to guide](https://qiskit.github.io/qiskit-addon-obp/how_tos/truncate_operator_terms.html).

In [11]:
# Run the same experiment but truncate observable terms with small coefficients
bp_obs_trunc, remaining_slices_trunc, metadata = backpropagate(
    observable,
    slices,
    operator_budget=op_budget,
    truncation_error_budget=truncation_error_budget,
)

# Recombine the slices remaining after backpropagation
bp_circuit_trunc = combine_slices(
    remaining_slices_trunc, include_barriers=False
)

print(f"Backpropagated {metadata.num_backpropagated_slices} slices.")
print(
    f"New observable has {len(bp_obs_trunc.paulis)} terms, which can be combined into {len(bp_obs_trunc.group_commuting(qubit_wise=True))} groups.\n"
    f"After truncation, the error in our observable is bounded by {metadata.accumulated_error(0):.3e}"
)
print(
    f"Note that backpropagating one more slice would result in {metadata.backpropagation_history[-1].num_paulis[0]} terms "
    f"across {metadata.backpropagation_history[-1].num_qwc_groups} groups."
)
print("The remaining circuit after backpropagation looks as follows:")
bp_circuit_trunc.draw("mpl", scale=0.6)

Backpropagated 7 slices.
New observable has 82 terms, which can be combined into 8 groups.
After truncation, the error in our observable is bounded by 3.266e-02
Note that backpropagating one more slice would result in 114 terms across 12 groups.
The remaining circuit after backpropagation looks as follows:


<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/5e8bae1a-ef18-4eb0-9d2a-1ac7bbdced3b-1.avif" alt="Output of the previous code cell" />

ด้านล่างนี้จะเห็นว่าเราย้อนกลับ 6 slice และ term ถูกรวมเป็น 6 กลุ่มแทนที่จะเป็น 8 กลุ่ม ซึ่งหมายความว่าการย้อนกลับอีก 1 slice จะทำให้จำนวน Pauli group เกิน 8 เราสามารถยืนยันว่าเป็นเช่นนั้นโดยตรวจสอบ metadata ที่ส่งคืน นอกจากนี้ในส่วนนี้การแปลง Circuit เป็นแบบ exact คือไม่มี term ของ observable ใหม่ $O'$ ที่ถูกตัดทอนออก Circuit และ operator ที่ย้อนกลับแล้วให้ผลลัพธ์ที่ตรงกันกับ Circuit และ operator ดั้งเดิมทุกประการ

In [ ]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)

In [13]:
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)

# Transpile original experiment
circuit_isa = pm.run(circuit)
observable_isa = observable.apply_layout(circuit_isa.layout)

# Transpile backpropagated experiment
bp_circuit_isa = pm.run(bp_circuit)
bp_obs_isa = bp_obs.apply_layout(bp_circuit_isa.layout)

# Transpile the backpropagated experiment with truncated observable terms
bp_circuit_trunc_isa = pm.run(bp_circuit_trunc)
bp_obs_trunc_isa = bp_obs_trunc.apply_layout(bp_circuit_trunc_isa.layout)

![Output of the previous code cell](../docs/images/tutorials/operator-back-propagation/extracted-outputs/65ec9cb1-a4ed-497b-a616-180e9659956f-1.avif)

ต่อไปเราจะระบุปัญหาเดิมด้วยข้อจำกัดเดิมบนขนาดของ observable ที่ output อย่างไรก็ตามคราวนี้เราจะจัดสรร error budget ให้แต่ละ slice โดยใช้ฟังก์ชัน [setup_budget](https://docs.quantum.ibm.com/api/qiskit-addon-obp/utils-truncating#setup_budget) Pauli term ที่มีสัมประสิทธิ์เล็กจะถูกตัดทอนออกจากแต่ละ slice จนกว่า error budget จะเต็ม และ budget ที่เหลือจะถูกบวกเพิ่มให้กับ budget ของ slice ถัดไป สังเกตว่าในกรณีนี้การแปลงเนื่องจากการย้อนกลับเป็นแบบประมาณ เนื่องจาก term บางส่วนใน operator ถูกตัดทอนออก

เพื่อเปิดใช้งานการตัดทอนนี้ เราต้องตั้งค่า error budget ดังนี้:

In [14]:
pub = (circuit_isa, observable_isa)
bp_pub = (bp_circuit_isa, bp_obs_isa)
bp_trunc_pub = (bp_circuit_trunc_isa, bp_obs_trunc_isa)

สังเกตว่าการจัดสรร error ขนาด `5e-3` ต่อ slice สำหรับการตัดทอน ทำให้เราสามารถลบ 1 slice เพิ่มเติมจาก Circuit ในขณะที่ยังอยู่ภายใน budget เดิมที่กำหนดไว้สำหรับ commuting Pauli group จำนวนแปดกลุ่มใน observable โดยค่าเริ่มต้น `backpropagate` ใช้ L1 norm ของสัมประสิทธิ์ที่ถูกตัดทอนเพื่อจำกัดข้อผิดพลาดรวมที่เกิดจากการตัดทอน สำหรับตัวเลือกอื่น ๆ ดู [how-to guide เกี่ยวกับการระบุ p_norm](https://qiskit.github.io/qiskit-addon-obp/how_tos/bound_error_using_p_norm.html)

ในตัวอย่างนี้ที่เราย้อนกลับ 7 slice ข้อผิดพลาดจากการตัดทอนรวมไม่ควรเกิน ``(5e-3 error/slice) * (7 slices) = 3.5e-2``
สำหรับการอภิปรายเพิ่มเติมเกี่ยวกับการกระจาย error budget ให้กับ slice ดู [how-to guide นี้](https://qiskit.github.io/qiskit-addon-obp/how_tos/truncate_operator_terms.html)

In [15]:
ideal_estimator = Estimator()

# Run the experiments using Estimator primitive to obtain the exact outcome
result_exact = (
    ideal_estimator.run([(circuit, observable)]).result()[0].data.evs.item()
)
print(f"Exact expectation value: {result_exact}")

Exact expectation value: 0.8871244838989416


We shall use <a href="/docs/guides/configure-error-mitigation">resilience_level</a> = 2 for this example.

In [ ]:
options = EstimatorOptions()
options.default_precision = 0.011
options.resilience_level = 2

estimator = EstimatorV2(mode=backend, options=options)

In [ ]:
job = estimator.run([pub, bp_pub, bp_trunc_pub])

### Step 4: Post-process and return result to desired classical format

In [ ]:
result_no_bp = job.result()[0].data.evs.item()
result_bp = job.result()[1].data.evs.item()
result_bp_trunc = job.result()[2].data.evs.item()

std_no_bp = job.result()[0].data.stds.item()
std_bp = job.result()[1].data.stds.item()
std_bp_trunc = job.result()[2].data.stds.item()

In [ ]:
print(
    f"Expectation value without backpropagation: {result_no_bp} ± {std_no_bp}"
)
print(f"Backpropagated expectation value: {result_bp} ± {std_bp}")
print(
    f"Backpropagated expectation value with truncation: {result_bp_trunc} ± {std_bp_trunc}"
)

Expectation value without backpropagation: 0.8033194665993642
Backpropagated expectation value: 0.8599808781259016
Backpropagated expectation value with truncation: 0.8868736004169483


In [ ]:
methods = [
    "No backpropagation",
    "Backpropagation",
    "Backpropagation w/ truncation",
]
values = [result_no_bp, result_bp, result_bp_trunc]
stds = [std_no_bp, std_bp, std_bp_trunc]

ax = plt.gca()
plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
plt.axhline(result_exact)
ax.set_ylim([0.6, 0.92])
plt.text(0.2, 0.895, "Exact result")
ax.set_ylabel(r"$M_Z$", fontsize=12)

Text(0, 0.5, '$M_Z$')

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/b444d8bc-c800-4aa3-9927-eb807e92194f-1.avif" alt="Output of the previous code cell" />

## Part B: Scale it up!

Let us now use Operator Backpropagation to study the dynamics of the Hamiltonian of a 50-qubit Heisenberg Spin Chain.

### Step 1: Map classical inputs to a quantum problem

We consider a 50-qubit Hamiltonian $\hat{\mathcal{H}}_{XYZ}$ for the scaled up problem with the same values for the $J$ and $h$ coefficients as in the small-scale example. The observable $M_Z = \frac{1}{N} \sum_{i=1}^N \langle Z_i \rangle$ is also the same as before. This problem is beyond classical brute-force simulation.

In [16]:
num_qubits = 50
layout = [(i - 1, i) for i in range(1, num_qubits)]

# Instantiate a CouplingMap object
coupling_map = CouplingMap(layout)
graphviz_draw(coupling_map.graph, method="circo")

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/47cb1ac7-44db-4f96-b49b-e889a920d83c-0.avif" alt="Output of the previous code cell" />

In [17]:
hamiltonian = generate_xyz_hamiltonian(
    coupling_map,
    coupling_constants=(np.pi / 8, np.pi / 4, np.pi / 2),
    ext_magnetic_field=(np.pi / 3, np.pi / 6, np.pi / 9),
)
print(hamiltonian)

SparsePauliOp(['IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIXXI', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIYYI', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZZI', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIXXIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIYYIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZZIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIXXIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIYYIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZZIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIXXIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIYYIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZZIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIXXIIIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIYYIIIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZZIIIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIXXIIIIIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIYYIIIIIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZZIIIIIIIIIII', 'IIIIIIIIIIII

In [18]:
observable = SparsePauliOp.from_sparse_list(
    [("Z", [i], 1 / num_qubits) for i in range(num_qubits)],
    num_qubits,
)
observable

SparsePauliOp(['IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZ', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZI', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZIIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZIIIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZIIIIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZIIIIIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZIIIIIIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZIIIIIIIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZIIIIIIIIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZIIIIIIIIIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZIIIIIIIIIIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZIIIIIIIIIIIIIIIII', 'IIIIIIIIIIII

### ขั้นตอนที่ 4: ประมวลผลหลังการรันและแปลงผลลัพธ์กลับเป็นรูปแบบ classical ที่ต้องการ

In [19]:
circuit = generate_time_evolution_circuit(
    hamiltonian,
    time=0.2,
    synthesis=LieTrotter(reps=4),
)
circuit.draw("mpl", style="iqp", fold=-1, scale=0.6)

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/b10d16cf-95da-42c0-9b47-b2e5a8516c82-0.avif" alt="Output of the previous code cell" />

### Step 2: Optimize problem for quantum hardware execution

In [20]:
slices = slice_by_gate_types(circuit)
print(f"Separated the circuit into {len(slices)} slices.")

Separated the circuit into 36 slices.


We specify the `max_error_per_slice` to be 0.005 as before. However, since the number of slices for this large-scale problem is much higher than the small scale problem, allowing an error of 0.005 per slice may end up creating a large overall backpropagation error. We can bound this by specifying `max_error_total` which bounds the total backpropagation error, and we set its value to 0.03 (which is roughly the same as in the small-scale example).

For this large-scale example, we allow a higher value for the number of commuting groups, and set it to 15.

In [21]:
op_budget = OperatorBudget(max_qwc_groups=15)
truncation_error_budget = setup_budget(
    max_error_total=0.03, max_error_per_slice=0.005
)

Let us first obtain the backpropagated circuit and observable without any truncation.

In [22]:
bp_obs, remaining_slices, metadata = backpropagate(
    observable, slices, operator_budget=op_budget
)
bp_circuit = combine_slices(remaining_slices)

print(f"Backpropagated {metadata.num_backpropagated_slices} slices.")
print(
    f"New observable has {len(bp_obs.paulis)} terms, which can be combined into {len(bp_obs.group_commuting(qubit_wise=True))} groups."
)
print(
    f"Note that backpropagating one more slice would result in {metadata.backpropagation_history[-1].num_paulis[0]} terms "
    f"across {metadata.backpropagation_history[-1].num_qwc_groups} groups."
)
print("The remaining circuit after backpropagation looks as follows:")
bp_circuit.draw("mpl", fold=-1, scale=0.6)

Backpropagated 7 slices.
New observable has 634 terms, which can be combined into 12 groups.
Note that backpropagating one more slice would result in 1246 terms across 27 groups.
The remaining circuit after backpropagation looks as follows:


<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/164e2f00-25b6-4cf1-98f8-8b2886f007ee-1.avif" alt="Output of the previous code cell" />

Now allowing for truncation, we obtain:

In [23]:
bp_obs_trunc, remaining_slices_trunc, metadata = backpropagate(
    observable,
    slices,
    operator_budget=op_budget,
    truncation_error_budget=truncation_error_budget,
)

# Recombine the slices remaining after backpropagation
bp_circuit_trunc = combine_slices(
    remaining_slices_trunc, include_barriers=False
)

print(f"Backpropagated {metadata.num_backpropagated_slices} slices.")
print(
    f"New observable has {len(bp_obs_trunc.paulis)} terms, which can be combined into {len(bp_obs_trunc.group_commuting(qubit_wise=True))} groups.\n"
    f"After truncation, the error in our observable is bounded by {metadata.accumulated_error(0):.3e}"
)
print(
    f"Note that backpropagating one more slice would result in {metadata.backpropagation_history[-1].num_paulis[0]} terms "
    f"across {metadata.backpropagation_history[-1].num_qwc_groups} groups."
)
print("The remaining circuit after backpropagation looks as follows:")
bp_circuit_trunc.draw("mpl", fold=-1, scale=0.6)

Backpropagated 10 slices.
New observable has 646 terms, which can be combined into 14 groups.
After truncation, the error in our observable is bounded by 2.998e-02
Note that backpropagating one more slice would result in 1226 terms across 29 groups.
The remaining circuit after backpropagation looks as follows:


<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/c05a85bc-e5ca-4e02-8c96-98b28811f335-1.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/operator-back-propagation/extracted-outputs/b444d8bc-c800-4aa3-9927-eb807e92194f-1.avif)
## Part B: ขยายขนาดขึ้น!
มาใช้ Operator Backpropagation เพื่อศึกษาพลวัตของ Hamiltonian ของ 50-Qubit Heisenberg Spin Chain กัน

### ขั้นตอนที่ 1: แปลงข้อมูลนำเข้าเชิงคลาสสิกเป็นปัญหาควอนตัม
เราพิจารณา Hamiltonian แบบ 50-Qubit $\hat{\mathcal{H}}_{XYZ}$ สำหรับปัญหาขนาดใหญ่ โดยใช้ค่า $J$ และ $h$ เหมือนกับตัวอย่างขนาดเล็ก observable $M_Z = \frac{1}{N} \sum_{i=1}^N \langle Z_i \rangle$ ก็เหมือนเดิมเช่นกัน ปัญหานี้เกินกำลังการจำลองแบบ brute-force ของคอมพิวเตอร์คลาสสิก

In [24]:
# Transpile original experiment
circuit_isa = pm.run(circuit)
observable_isa = observable.apply_layout(circuit_isa.layout)

# Transpile the backpropagated experiment
bp_circuit_isa = pm.run(bp_circuit)
bp_obs_isa = bp_obs_trunc.apply_layout(bp_circuit_isa.layout)

# Transpile the backpropagated experiment with truncated observable terms
bp_circuit_trunc_isa = pm.run(bp_circuit_trunc)
bp_obs_trunc_isa = bp_obs_trunc.apply_layout(bp_circuit_trunc_isa.layout)

In [25]:
print(
    f"2-qubit depth of original circuit: {circuit_isa.depth(lambda x:x.operation.num_qubits==2)}"
)
print(
    f"2-qubit depth of backpropagated circuit: {bp_circuit_isa.depth(lambda x:x.operation.num_qubits==2)}"
)
print(
    f"2-qubit depth of backpropagated circuit with truncation: {bp_circuit_trunc_isa.depth(lambda x:x.operation.num_qubits==2)}"
)

2-qubit depth of original circuit: 48
2-qubit depth of backpropagated circuit: 40
2-qubit depth of backpropagated circuit with truncation: 36


### Step 3: Execute using Qiskit primitives

In [26]:
pubs = [
    (circuit_isa, observable_isa),
    (bp_circuit_isa, bp_obs_isa),
    (bp_circuit_trunc_isa, bp_obs_trunc_isa),
]

In [27]:
options = EstimatorOptions()
options.default_precision = 0.01
options.resilience_level = 2
options.resilience.zne.noise_factors = [1, 1.2, 1.4]
options.resilience.zne.extrapolator = ["linear"]

estimator = EstimatorV2(mode=backend, options=options)

In [ ]:
job = estimator.run(pubs)

สำหรับปัญหาขนาดใหญ่นี้ เราพิจารณาเวลาวิวัฒนาการเป็น $0.2$ โดยมี $4$ ขั้นตอน trotter ปัญหานี้ถูกเลือกให้เกินกำลังการจำลองแบบ brute-force เชิงคลาสสิก แต่สามารถจำลองได้ด้วยวิธี tensor network ซึ่งทำให้เราตรวจสอบผลลัพธ์ที่ได้จาก backpropagation บนคอมพิวเตอร์ควอนตัมกับผลลัพธ์ที่แม่นยำได้

ค่าคาดหวังแบบ ideal สำหรับปัญหานี้ ซึ่งได้จากการจำลอง tensor network คือ $\simeq 0.89$

In [ ]:
result_no_bp = job.result()[0].data.evs.item()
result_bp = job.result()[1].data.evs.item()
result_bp_trunc = job.result()[2].data.evs.item()

In [ ]:
print(f"Expectation value without backpropagation: {result_no_bp}")
print(f"Backpropagated expectation value: {result_bp}")
print(f"Backpropagated expectation value with truncation: {result_bp_trunc}")

Expectation value without backpropagation: 0.7887194658035515
Backpropagated expectation value: 0.9532818300978584
Backpropagated expectation value with truncation: 0.8913400398926913


In [ ]:
methods = [
    "No backpropagation",
    "Backpropagation",
    "Backpropagation w/ truncation",
]
values = [result_no_bp, result_bp, result_bp_trunc]

ax = plt.gca()
plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
plt.axhline(0.89)
ax.set_ylim([0.6, 0.98])
plt.text(0.2, 0.895, "Exact result")
ax.set_ylabel(r"$M_Z$", fontsize=12)

Text(0, 0.5, '$M_Z$')

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/047d448f-aebf-45ff-a81b-83b2d5ca866d-1.avif" alt="Output of the previous code cell" />

เราระบุ `max_error_per_slice` เป็น 0.005 เหมือนเดิม อย่างไรก็ตาม เนื่องจากจำนวน slice ในปัญหาขนาดใหญ่นี้มากกว่าปัญหาขนาดเล็กมาก การอนุญาตให้มีข้อผิดพลาด 0.005 ต่อ slice อาจทำให้เกิดข้อผิดพลาดรวมจาก backpropagation ที่มาก เราจึงกำหนด `max_error_total` เพื่อควบคุมข้อผิดพลาด backpropagation รวม โดยตั้งค่าเป็น 0.03 (ซึ่งใกล้เคียงกับตัวอย่างขนาดเล็ก)

สำหรับตัวอย่างขนาดใหญ่นี้ เราอนุญาตให้ใช้จำนวน commuting group มากขึ้น โดยตั้งค่าเป็น 15